In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# TCGA-LIHC RNA-seq
# Step 1：Raw Counts → QC → 低表达过滤 → 标准化 → PCA
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 文件路径
# ------------------------------------------------------------

clinical_file = _hcc_path('LIHC_Clinical_Survival_Final_371.csv')
rna_file = _hcc_path('LIHC_RNA_RawCounts_Final_371.tsv')

# 输出文件夹
output_dir = "LIHC_RNA_Step1"
os.makedirs(output_dir, exist_ok=True)

print("工作目录：", os.getcwd())
print("输出目录：", output_dir)


In [ ]:
# ============================================================
# 读取数据
# ============================================================

clinical = pd.read_csv(clinical_file)

rna = pd.read_csv(
    rna_file,
    sep="\t"
)

print("=" * 60)
print("Clinical")
print("=" * 60)
print("维度：", clinical.shape)
print("前5列：", clinical.columns[:5].tolist())

print()

print("=" * 60)
print("RNA Raw Counts")
print("=" * 60)
print("维度：", rna.shape)
print("前5列：", rna.columns[:5].tolist())

In [ ]:
# ============================================================
# Gene Symbol QC
# ============================================================

gene_col = "Gene_Symbol"

if gene_col not in rna.columns:
    raise ValueError(f"RNA 文件中没有找到 {gene_col} 列！")

print("Gene Symbol 总数：", rna[gene_col].nunique())
print("RNA 总行数：", len(rna))

duplicate_genes = rna[gene_col][rna[gene_col].duplicated()].unique()

print("重复 Gene Symbol 数：", len(duplicate_genes))

if len(duplicate_genes) > 0:
    print("前20个重复基因：")
    print(duplicate_genes[:20])
else:
    print("✓ Gene Symbol 无重复")

In [ ]:
# ============================================================
# Sample_ID Matching QC
# ============================================================

clinical_samples = clinical["Sample_ID"].astype(str).tolist()

rna_samples = [
    col for col in rna.columns
    if col != gene_col
]

clinical_set = set(clinical_samples)
rna_set = set(rna_samples)

matched = sorted(clinical_set & rna_set)

clinical_only = sorted(clinical_set - rna_set)
rna_only = sorted(rna_set - clinical_set)

print("=" * 60)
print("Sample Matching QC")
print("=" * 60)

print("Clinical 样本数：", len(clinical_set))
print("RNA 样本数：", len(rna_set))
print("成功匹配：", len(matched))
print("Clinical 有 / RNA 无：", len(clinical_only))
print("RNA 有 / Clinical 无：", len(rna_only))

if len(clinical_only) > 0:
    print("\nClinical 有但 RNA 没有：")
    print(clinical_only[:20])

if len(rna_only) > 0:
    print("\nRNA 有但 Clinical 没有：")
    print(rna_only[:20])

In [ ]:
# ============================================================
# 按 Clinical Sample_ID 顺序排列 RNA
# ============================================================

if len(matched) != len(clinical_set) or len(matched) != len(rna_set):
    raise ValueError(
        "Clinical 和 RNA 没有完全匹配，请先解决 Sample_ID 问题！"
    )

ordered_samples = clinical["Sample_ID"].astype(str).tolist()

rna = rna[[gene_col] + ordered_samples]

print("✓ RNA 已按照 Clinical Sample_ID 顺序重新排列")

print("\n前10个 RNA Sample_ID：")
print(rna.columns[1:11].tolist())

print("\n前10个 Clinical Sample_ID：")
print(clinical["Sample_ID"].head(10).tolist())

In [ ]:
# ============================================================
# RNA 数值 QC
# ============================================================

rna_numeric = rna.iloc[:, 1:].apply(
    pd.to_numeric,
    errors="coerce"
)

non_numeric = rna_numeric.isna().sum().sum()

print("非数值单元格数量：", non_numeric)

if non_numeric > 0:
    raise ValueError("RNA 中存在非数值单元格，请检查原始文件！")

# 转成整数
rna_numeric = rna_numeric.astype(np.int64)

print("✓ 所有 RNA 表达值均为数值")
print("✓ Raw Counts 已转换为整数")

In [ ]:
# ============================================================
# 检查负数
# ============================================================

negative_count = (rna_numeric < 0).sum().sum()

print("负数数量：", negative_count)

if negative_count > 0:
    raise ValueError("发现负数 Counts，请检查数据！")
else:
    print("✓ 没有负数")

In [ ]:
# ============================================================
# Gene Expression QC
# ============================================================

gene_names = rna[gene_col].astype(str)

gene_total_counts = rna_numeric.sum(axis=1)

gene_detected_samples = (rna_numeric > 0).sum(axis=1)

gene_mean_counts = rna_numeric.mean(axis=1)

gene_qc = pd.DataFrame({
    "Gene_Symbol": gene_names,
    "Total_Counts": gene_total_counts,
    "Mean_Counts": gene_mean_counts,
    "Detected_Samples": gene_detected_samples
})

print(gene_qc.head())

print("\n检测到表达的基因：",
      (gene_detected_samples > 0).sum())

print("全0基因：",
      (gene_detected_samples == 0).sum())

In [ ]:
# ============================================================
# Low Expression Filtering
# 至少20%的样本 Counts >= 10
# ============================================================

n_samples = rna_numeric.shape[1]

#0.20处可修改低表达保留量
min_samples = int(np.ceil(n_samples * 0.20))

print("样本数：", n_samples)
print("过滤阈值：")
print(f"至少 {min_samples} 个样本 Counts >= 10")

keep_gene = (
    (rna_numeric >= 10).sum(axis=1) >= min_samples
)

print("\n过滤前基因数：", len(keep_gene))
print("保留基因数：", keep_gene.sum())
print("删除基因数：", (~keep_gene).sum())

print(
    "保留比例：",
    round(keep_gene.mean() * 100, 2),
    "%"
)

In [ ]:
# ============================================================
# Filtered Raw Counts
# ============================================================

filtered_counts = pd.concat(
    [
        gene_names[keep_gene].reset_index(drop=True),
        rna_numeric.loc[keep_gene].reset_index(drop=True)
    ],
    axis=1
)

filtered_counts.columns = [gene_col] + ordered_samples

print("过滤后的表达矩阵：")
print(filtered_counts.shape)

print("\n前5行：")
display(filtered_counts.head())

In [ ]:
filtered_counts_file = os.path.join(
    output_dir,
    "LIHC_RNA_FilteredCounts_371.tsv"
)

filtered_counts.to_csv(
    filtered_counts_file,
    sep="\t",
    index=False
)

print("✓ 已保存：")
print(filtered_counts_file)

In [ ]:
# ============================================================
# Sample sequencing depth
# ============================================================

sample_library_size = filtered_counts.iloc[:, 1:].sum(axis=0)

library_df = pd.DataFrame({
    "Sample_ID": sample_library_size.index,
    "Library_Size": sample_library_size.values
})

print(library_df.describe())

In [ ]:
plt.figure(figsize=(12, 5))

plt.hist(
    np.log10(library_df["Library_Size"] + 1),
    bins=30
)

plt.xlabel("log10(Library Size + 1)")
plt.ylabel("Number of Samples")
plt.title("TCGA-LIHC RNA-seq Library Size")

plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "01_Library_Size.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# CPM normalization
# ============================================================

count_matrix = filtered_counts.iloc[:, 1:].astype(float)

library_sizes = count_matrix.sum(axis=0)

cpm = count_matrix.div(
    library_sizes,
    axis=1
) * 1e6

print("CPM matrix：", cpm.shape)

In [ ]:
# ============================================================
# log2(CPM + 1)
# ============================================================

log_cpm = np.log2(cpm + 1)

print("log2(CPM + 1) matrix：", log_cpm.shape)

display(log_cpm.iloc[:5, :5])

In [ ]:
# ============================================================
# 保存 normalized expression matrix
# ============================================================

normalized_expression = pd.concat(
    [
        filtered_counts[gene_col],
        log_cpm.reset_index(drop=True)
    ],
    axis=1
)

normalized_expression.columns = [
    gene_col
] + ordered_samples

normalized_file = os.path.join(
    output_dir,
    "LIHC_RNA_Normalized_371.tsv"
)

normalized_expression.to_csv(
    normalized_file,
    sep="\t",
    index=False
)

print("✓ 标准化表达矩阵已保存：")
print(normalized_file)

print("\n最终矩阵维度：")
print(normalized_expression.shape)

In [ ]:
plt.figure(figsize=(14, 6))

for sample in plot_samples:
    sns.kdeplot(
        log_cpm[sample],
        alpha=0.25
    )

plt.xlabel("log2(CPM + 1)")
plt.ylabel("Density")
plt.title("RNA-seq Expression Distribution")

plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "02_Expression_Distribution.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# PCA
# ============================================================

# PCA需要：
# 样本 × 基因
pca_input = log_cpm.T

print("PCA 输入维度：", pca_input.shape)

# 标准化每个基因
pca_scaled = StandardScaler().fit_transform(pca_input)

pca = PCA(n_components=2)

pca_result = pca.fit_transform(pca_scaled)

pca_df = pd.DataFrame(
    pca_result,
    columns=["PC1", "PC2"],
    index=pca_input.index
)

print("PC1 explained variance:",
      round(pca.explained_variance_ratio_[0] * 100, 2), "%")

print("PC2 explained variance:",
      round(pca.explained_variance_ratio_[1] * 100, 2), "%")

In [ ]:
plt.figure(figsize=(8, 7))

plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    s=35,
    alpha=0.8
)

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)"
)

plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)"
)

plt.title("TCGA-LIHC RNA-seq PCA")

plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "03_PCA.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# PCA + Clinical annotation
# ============================================================

pca_clinical = pca_df.reset_index()

pca_clinical = pca_clinical.rename(
    columns={"index": "Sample_ID"}
)

pca_clinical = pca_clinical.merge(
    clinical,
    on="Sample_ID",
    how="left"
)

print(pca_clinical.shape)

display(
    pca_clinical[
        ["Sample_ID", "PC1", "PC2", "Stage", "Gender", "Age"]
    ].head()
)

In [ ]:
# ============================================================
# PCA by Stage
# ============================================================

plt.figure(figsize=(9, 7))

stages = pca_clinical["Stage"].dropna().unique()

for stage in sorted(stages):
    
    sub = pca_clinical[
        pca_clinical["Stage"] == stage
    ]
    
    plt.scatter(
        sub["PC1"],
        sub["PC2"],
        s=40,
        alpha=0.75,
        label=stage
    )

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)"
)

plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)"
)

plt.title("TCGA-LIHC PCA by Clinical Stage")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "04_PCA_by_Stage.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# Step 1 Summary
# ============================================================

summary = {
    "Clinical samples": len(clinical_set),
    "RNA samples": len(rna_set),
    "Matched samples": len(matched),
    "Original genes": len(rna),
    "Filtered genes": int(keep_gene.sum()),
    "Removed genes": int((~keep_gene).sum()),
    "Filtering threshold": f"Counts >= 10 in >= {min_samples} samples",
    "Final expression matrix": str(normalized_expression.shape)
}

print("=" * 60)
print("TCGA-LIHC RNA-seq STEP 1 完成")
print("=" * 60)

for k, v in summary.items():
    print(f"{k}: {v}")

print("\n输出文件：")
print("1.", filtered_counts_file)
print("2.", normalized_file)

print("\nQC 图：")
print("1. 01_Library_Size.png")
print("2. 02_Expression_Distribution.png")
print("3. 03_PCA.png")
print("4. 04_PCA_by_Stage.png")

print("=" * 60)